# Utility comparison on multi-MNIST (linear network)

Train a fully-connected linear network on 2-task multi-MNIST and track
per-weight EMA utilities under four different utility functions.

Plots:
- Line plot of average utility over time, split by within-task vs cross-task connections.
- Distribution plot at end of training, split by within-task vs cross-task.
- Line plot of "overlap count" — number of connections whose utility lies between
  the highest cross-task utility and the lowest within-task utility (lower = better separation).

In [44]:
import os
import sys

# Make the repo's `data` module importable. `data.py` lives in
# phd/structure_search/, so we need that directory on the path.
REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS    # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS      # 20

print('JAX device:', jax.devices()[0])

JAX device: cuda:0


## Data loading

In [45]:
def load_data():
    """Load MNIST and standardize per-pixel (mean 0, std 1).

    Returns (images, labels) as JAX arrays. Each image is shape (784,).
    Compose two of these to make a multi-MNIST sample.
    """
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, 'dtype', images.dtype)
print('labels:', labels.shape, 'min/max:', int(labels.min()), int(labels.max()))

images: (60000, 784) dtype float32
labels: (60000,) min/max: 0 9


## Utility functions

Each utility takes (W, M, x, y, logits) and returns a per-weight utility array
of the same shape as W. Sign convention: **U > 0 means removing the weight
hurts loss → keep**.

Conventions:
- `W` shape `(IN, OUT)`. `M` is a binary mask (also `(IN, OUT)`).
- `logits = x @ W`, shape `(OUT,)`.
- For multi-MNIST, `OUT = N_TASKS · NUM_CLASSES`. Reshape `(N_TASKS, NUM_CLASSES)`
  for per-task softmax.

In [46]:
def signed_utility(W, M, x, y, logits):
    """L1-loss LOO on raw logits: u = |e + c| - |e|, with e = onehot - logits, c = x_j * W[j,k]."""
    y_onehot = jax.nn.one_hot(y, num_classes=NUM_CLASSES).reshape(-1)   # (OUT,)
    e = y_onehot - logits                                                # (OUT,)
    c = x[:, None] * W                                                   # (IN, OUT)
    u = jnp.abs(e[None, :] + c) - jnp.abs(e[None, :])
    return u * M


def softmax_ce_utility(W, M, x, y, logits):
    """Closed-form per-weight softmax-CE LOO. Per-task softmax.

    For weight (j, k): if we remove the weight, logit_k changes by d = -x_j*W[j,k].
    Within task t_k, the softmax-CE for the correct class changes by
        ΔCE = -d * y_match[k] + log((1-p[k]) + p[k] * e^d)
    and other tasks are unaffected.

    Numerically stable: the obvious form log1p(p * expm1(d)) NaNs once p
    saturates (p → 1 means expm1(d) floors at -1.0 in float32, log1p(-1.0)
    = -inf). We rewrite as
        log((1-p) + p*e^d) = logaddexp(log(1-p), d + log(p))
    where log(1-p_k) is computed as logsumexp(log_p_{j != k}) so the
    saturated case (log_p_k = 0 exactly) still produces a finite, accurate
    value derived from the other classes' log probabilities.
    """
    logits_per_task = logits.reshape(N_TASKS, NUM_CLASSES)
    log_p_pt = jax.nn.log_softmax(logits_per_task, axis=-1)   # (T, C)
    # log(1 - p_{t,k}) = logsumexp over j != k of log_p_{t, j}.
    diag = jnp.eye(NUM_CLASSES, dtype=jnp.bool_)              # (C, C)
    log_p_off = jnp.where(diag[None, :, :],
                          -jnp.inf,
                          log_p_pt[:, None, :])               # (T, C, C)
    log_1mp_pt = jax.scipy.special.logsumexp(log_p_off, axis=-1)  # (T, C)
    log_p = log_p_pt.reshape(-1)                              # (OUT,)
    log_1mp = log_1mp_pt.reshape(-1)                          # (OUT,)
    y_match = jax.nn.one_hot(y, num_classes=NUM_CLASSES).reshape(-1)
    d = -x[:, None] * W                                       # (IN, OUT)
    a = log_1mp[None, :]                                      # (1, OUT)
    b = d + log_p[None, :]                                    # (IN, OUT)
    return (-d * y_match[None, :] + jnp.logaddexp(a, b)) * M


def weight_magnitude_utility(W, M, x, y, logits):
    """|W|. Doesn't depend on the sample; EMA'd over time it just trails |W|."""
    return jnp.abs(W) * M


def contribution_utility(W, M, x, y, logits):
    """|x_j| * |W[j,k]|: instantaneous magnitude of this weight's contribution."""
    return jnp.abs(x[:, None]) * jnp.abs(W) * M


UTILITY_FNS = {
    'signed':           signed_utility,
    'softmax_ce':       softmax_ce_utility,
    'weight_magnitude': weight_magnitude_utility,
    'contribution':     contribution_utility,
}

## Training function

Edit this cell freely. The function is JIT-compatible so wrap with `jax.jit` for speed
(see the next cell). It runs `n_steps` of online SGD, EMA-tracks the four utilities,
and returns a dict of per-snapshot arrays plus the final state.

In [47]:
def train(W_init, images, labels, *,
          lr=2**-9,
          beta=0.998,
          n_steps=225_000,
          snapshot_every=1000,
          permute_period=0,
          seed=0):
    """Train a linear network on multi-MNIST and snapshot all four utility EMAs.

    Args:
      W_init: initial weights, shape (INPUT_DIM, OUTPUT_DIM).
      images, labels: standardized MNIST.
      lr: SGD learning rate.
      beta: EMA decay for the four utilities.
      n_steps: total training steps.
      snapshot_every: cadence of utility-EMA snapshots (also avg-loss bins).
      permute_period: every N steps, permute one task's class labels.
        0 = stationary (no permutation).
      seed: PRNG seed.

    Returns:
      dict with keys:
        'steps':        (n_snapshots,)  -- step index at each snapshot
        'avg_loss':     (n_snapshots,)  -- mean loss over each snapshot interval
        'W':            (n_snapshots, IN, OUT)  -- weight snapshot
        'U_<name>':     (n_snapshots, IN, OUT)  -- bias-corrected EMA per utility
        'final_W':      (IN, OUT)
    """
    n_snapshots = n_steps // snapshot_every
    M = jnp.ones_like(W_init, dtype=W_init.dtype)
    util_names = list(UTILITY_FNS.keys())
    perm0 = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1 = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    Us = {name: jnp.zeros_like(W_init) for name in util_names}
    init_carry = (W_init, *[Us[n] for n in util_names], perm0, perm1,
                  jnp.array(0, dtype=jnp.int32))

    def loss_fn(W, x, y):
        logits = x @ W
        logits_per_task = logits.reshape(N_TASKS, NUM_CLASSES)
        lp = jax.nn.log_softmax(logits_per_task, axis=-1)
        return -jnp.mean(jnp.sum(jax.nn.one_hot(y, NUM_CLASSES) * lp, axis=-1))

    def make_sample(key):
        k1, k2 = jax.random.split(key)
        idx1 = jax.random.randint(k1, (), 0, images.shape[0])
        idx2 = jax.random.randint(k2, (), 0, images.shape[0])
        x = jnp.concatenate([images[idx1], images[idx2]])
        y_raw = jnp.array([labels[idx1], labels[idx2]])
        return x, y_raw

    def step_fn(carry, key):
        W, U_signed, U_softmax, U_mag, U_contrib, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])

        loss, grad = jax.value_and_grad(loss_fn)(W, x, y)
        logits = x @ W

        # Instantaneous utilities (computed BEFORE the W update — uses current W).
        u_signed   = signed_utility(W, M, x, y, logits)
        u_softmax  = softmax_ce_utility(W, M, x, y, logits)
        u_mag      = weight_magnitude_utility(W, M, x, y, logits)
        u_contrib  = contribution_utility(W, M, x, y, logits)

        U_signed   = beta * U_signed   + (1.0 - beta) * u_signed
        U_softmax  = beta * U_softmax  + (1.0 - beta) * u_softmax
        U_mag      = beta * U_mag      + (1.0 - beta) * u_mag
        U_contrib  = beta * U_contrib  + (1.0 - beta) * u_contrib

        W = W - lr * grad
        t = t + 1

        # Optional task-class label permutation (every permute_period steps,
        # pick a task at random and permute its class assignments).
        if permute_period > 0:
            should_perm = (t >= permute_period) & (t % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)

        return (W, U_signed, U_softmax, U_mag, U_contrib, perm0, perm1, t), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W, U_signed, U_softmax, U_mag, U_contrib, _, _, t = carry
        bias_corr = 1.0 / jnp.maximum(1.0 - beta ** t.astype(jnp.float32), 1e-12)
        snap = dict(
            step=t,
            avg_loss=losses.mean(),
            W=W,
            U_signed=U_signed * bias_corr,
            U_softmax=U_softmax * bias_corr,
            U_weight_magnitude=U_mag * bias_corr,
            U_contribution=U_contrib * bias_corr,
        )
        return carry, snap

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_snapshots)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W'] = jax.device_get(final_carry[0])
    return snaps

## Run training

Configure hyperparameters here. The first run JIT-compiles (slower); subsequent runs
with the same `n_steps`, `snapshot_every`, `permute_period` use the cache.

In [48]:
train_jit = jax.jit(
    train,
    static_argnames=('n_steps', 'snapshot_every', 'permute_period', 'lr', 'beta', 'seed'),
)

W_init = jnp.zeros((INPUT_DIM, OUTPUT_DIM), dtype=jnp.float32)

snaps = train_jit(
    W_init, images, labels,
    lr=2**-9,
    beta=0.998,
    n_steps=225_000,
    snapshot_every=1_000,
    permute_period=0,    # 0 = stationary; e.g. 4000 to permute every 4k steps
    seed=0,
)

print('snapshots:', len(snaps['avg_loss']))
print('final loss (avg over last interval):', float(snaps['avg_loss'][-1]))

snapshots: 225
final loss (avg over last interval): 0.3219233453273773


## Plotting

Helpers for cross/within masks and the three plot types.

In [49]:
# Cross-task / within-task masks for connections (IN, OUT).
# Within-task: input's task index == output's task index.
input_task = np.arange(INPUT_DIM) // INPUT_PER_TASK         # (IN,)
output_task = np.arange(OUTPUT_DIM) // NUM_CLASSES          # (OUT,)
within_mask = (input_task[:, None] == output_task[None, :]) # (IN, OUT) bool
cross_mask = ~within_mask
UTILITY_KEYS = ['U_signed', 'U_softmax', 'U_weight_magnitude', 'U_contribution']
print(f'within-task connections: {within_mask.sum()}, cross-task: {cross_mask.sum()}')

within-task connections: 15680, cross-task: 15680


In [50]:
WITHIN_COLOR = '#1f77b4'   # blue
CROSS_COLOR  = '#d62728'   # red


def plot_avg_utility_over_time(snaps, utility_keys=UTILITY_KEYS):
    """For each utility, plot the mean over within-task and cross-task connections at each snapshot."""
    steps = np.asarray(snaps['step']) if 'step' in snaps else np.arange(len(snaps['avg_loss']))
    fig = make_subplots(rows=1, cols=len(utility_keys), subplot_titles=utility_keys,
                        shared_yaxes=False)
    for i, name in enumerate(utility_keys, start=1):
        U = np.asarray(snaps[name])  # (n_snap, IN, OUT)
        within_mean = U[:, within_mask].mean(axis=-1)
        cross_mean  = U[:, cross_mask].mean(axis=-1)
        showlegend = (i == 1)  # one shared legend
        fig.add_trace(go.Scatter(
            x=steps, y=within_mean, mode='lines', name='within-task',
            line=dict(color=WITHIN_COLOR), showlegend=showlegend,
            legendgroup='within'), row=1, col=i)
        fig.add_trace(go.Scatter(
            x=steps, y=cross_mean, mode='lines', name='cross-task',
            line=dict(color=CROSS_COLOR), showlegend=showlegend,
            legendgroup='cross'), row=1, col=i)
        fig.update_xaxes(title_text='step', row=1, col=i)
    fig.update_yaxes(title_text='mean utility', row=1, col=1)
    fig.update_layout(title='Average utility over time (per task type)',
                      height=420, width=320 * len(utility_keys))
    fig.show()
    return fig

In [51]:
def plot_final_distributions(snaps, utility_keys=UTILITY_KEYS, bins=80):
    """Histograms of per-connection utility at the final snapshot, split within/cross.

    X-axis bounds: inner-95% range (P2.5 to P97.5 over all weights for that utility),
    extended by 10% of the inner range on each side. Outliers beyond are clipped.
    """
    fig = make_subplots(rows=1, cols=len(utility_keys), subplot_titles=utility_keys,
                        shared_yaxes=False)
    for i, name in enumerate(utility_keys, start=1):
        U_final = np.asarray(snaps[name][-1])
        within_vals = U_final[within_mask]
        cross_vals = U_final[cross_mask]
        all_vals = np.concatenate([within_vals, cross_vals])
        lo = np.percentile(all_vals, 2.5)
        hi = np.percentile(all_vals, 97.5)
        width = hi - lo
        if width <= 0:
            width = max(abs(lo), abs(hi), 1.0) * 0.1
        xmin = lo - 0.1 * width
        xmax = hi + 0.1 * width
        bin_size = (xmax - xmin) / bins
        showlegend = (i == 1)
        fig.add_trace(go.Histogram(
            x=cross_vals, opacity=0.55, name='cross-task',
            marker_color=CROSS_COLOR,
            xbins=dict(start=xmin, end=xmax, size=bin_size),
            showlegend=showlegend, legendgroup='cross'), row=1, col=i)
        fig.add_trace(go.Histogram(
            x=within_vals, opacity=0.55, name='within-task',
            marker_color=WITHIN_COLOR,
            xbins=dict(start=xmin, end=xmax, size=bin_size),
            showlegend=showlegend, legendgroup='within'), row=1, col=i)
        fig.update_xaxes(range=[xmin, xmax], title_text='utility', row=1, col=i)
    fig.update_yaxes(title_text='# connections', row=1, col=1)
    fig.update_layout(barmode='overlay', title='Final utility distributions',
                      height=420, width=320 * len(utility_keys))
    fig.show()
    return fig

In [52]:
def plot_separation_metric(snaps, utility_keys=UTILITY_KEYS, tail_pct=98):
    """Number of connections whose utility lies between the cross-task
    high-tail percentile and the within-task low-tail percentile (i.e.,
    the overlap region). Lower = cleaner separation, 0 = perfectly separated.

    `tail_pct=98` uses the 98th percentile of cross-task utilities as the
    upper threshold and the 2nd percentile of within-task utilities as the
    lower threshold (more robust to outliers than max/min).
    """
    steps = np.asarray(snaps['step']) if 'step' in snaps else np.arange(len(snaps['avg_loss']))
    lo_pct = 100.0 - tail_pct
    fig = go.Figure()
    for name in utility_keys:
        U = np.asarray(snaps[name])
        cross_vals = U[:, cross_mask]
        within_vals = U[:, within_mask]
        cross_high = np.percentile(cross_vals, tail_pct, axis=-1)
        within_low = np.percentile(within_vals, lo_pct, axis=-1)
        in_overlap = (U >= within_low[:, None, None]) & (U <= cross_high[:, None, None])
        counts = in_overlap.sum(axis=(1, 2))
        fig.add_trace(go.Scatter(x=steps, y=counts, mode='lines', name=name))
    fig.update_layout(
        title=f'Separation metric (tail_pct={tail_pct})',
        xaxis_title='step',
        yaxis_title=f'# connections in overlap [P{lo_pct:g} within, P{tail_pct:g} cross]',
        width=900, height=450,
    )
    fig.show()
    return fig


def plot_loss_over_time(snaps):
    """Per-snapshot mean training loss."""
    steps = np.asarray(snaps['step']) if 'step' in snaps else np.arange(len(snaps['avg_loss']))
    fig = go.Figure(go.Scatter(x=steps, y=np.asarray(snaps['avg_loss']),
                               mode='lines', name='loss'))
    fig.update_layout(title='Training loss',
                      xaxis_title='step',
                      yaxis_title='mean loss over snapshot interval',
                      width=800, height=380)
    fig.show()
    return fig

In [53]:
plot_loss_over_time(snaps)
plot_avg_utility_over_time(snaps)
plot_final_distributions(snaps)
plot_separation_metric(snaps)